# 03 - Class-incremental comparison: fixed linear vs expanding vs fixed-capacity polynomial heads

Ten classes arrive in five two-class experiences; one ten-output softmax head is trained on the current experience
plus a fixed per-class replay memory, and only classes introduced so far compete (future output weights are never
trained). Three conditions share task order, replay indices, seed, optimiser and epochs, and differ only in
polynomial degree: `fixed_linear` (degree 1), `expanding` (1, 2, 3, 3, 3, new coefficients start at zero) and
`fixed_capacity` (degree 3 throughout). Protocol details: `src/polynomial_cl/class_incremental/experiment.py`.

Two datasets: ten Gaussian classes on a ring in the plane, and Split MNIST under a fixed 16-dimensional random
projection (not fitted to data). Nothing here assumes expansion helps.

In [ ]:
import importlib.util
import pathlib
import sys

if importlib.util.find_spec("polynomial_cl") is None:  # not pip-installed: use src/
    sys.path.insert(0, str(pathlib.Path("..").resolve() / "src"))

import matplotlib.pyplot as plt
import numpy as np

from polynomial_cl import metrics
from polynomial_cl.class_incremental import (
    CONDITIONS,
    TASK_ORDER,
    load_projected_mnist,
    make_feature_data,
    make_synthetic_2d,
    make_task_plan,
    run_comparison,
    run_condition,
)
from polynomial_cl.class_incremental.mnist import FILES

%matplotlib inline
plt.rcParams.update({"figure.dpi": 80})

SEEDS = [0, 1, 2, 3, 4]
EPOCHS, LR, BATCH_SIZE, MEMORY_PER_CLASS = 3, 0.1, 128, 200
MNIST_DIR = pathlib.Path("..") / "data" / "mnist"
TRAIN = {"epochs": EPOCHS, "learning_rate": LR, "batch_size": BATCH_SIZE}
SETTINGS = {**TRAIN, "memory_per_class": MEMORY_PER_CLASS}


def summary_table(result):
    rows = [{"condition": c, **{m: f"{s['mean']:.3f}+/-{s['std']:.3f}" for m, s in stats.items()}}
            for c, stats in result["summary"].items()]
    print(metrics.format_table(rows))


def plot_matrices(result, title, seed=0):
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, condition in zip(axes, CONDITIONS):
        run = next(r for r in result["runs"] if r["seed"] == seed and r["condition"] == condition)
        matrix = np.array([[np.nan if v is None else v for v in row] for row in run["accuracy_matrix"]])
        image = ax.imshow(matrix, vmin=0, vmax=1, cmap="viridis")
        ax.set_title(condition)
        ax.set_xlabel("experience being evaluated")
        ax.set_ylabel("after training on experience")
        ax.set_xticks(range(5), range(1, 6))
        ax.set_yticks(range(5), range(1, 6))
    fig.colorbar(image, ax=axes, label="accuracy")
    fig.suptitle(f"{title}: accuracy matrices (seed {seed})")
    plt.show()

## Synthetic ring (2-D)

In [ ]:
x_tr, y_tr, x_te, y_te, means = make_synthetic_2d(0)
fig, ax = plt.subplots(figsize=(5, 5))
for task, classes in enumerate(TASK_ORDER):
    mask = np.isin(y_tr, classes)
    ax.scatter(x_tr[mask, 0], x_tr[mask, 1], s=8, alpha=0.6, label=f"experience {task + 1}: classes {classes}")
ax.set_aspect("equal"); ax.legend(fontsize=7); ax.set_title("Training data by experience")
plt.show()

synthetic = make_feature_data(x_tr, y_tr, x_te, y_te)
synthetic_result = run_comparison(synthetic, SEEDS, **SETTINGS)
summary_table(synthetic_result)
plot_matrices(synthetic_result, "Synthetic ring")

### Does the result depend on the training budget?

The ring classes have equal isotropic Gaussians, so a *linear* softmax head already reaches the Bayes-optimal
boundaries: this dataset cannot reward extra capacity, only faster optimisation. With a small epoch budget the
linear head has simply not finished learning each new class; the gap should close as epochs grow.

In [ ]:
rows = []
for epochs in (1, 3, 10, 30, 100):
    result = run_comparison(synthetic, SEEDS[:3], **{**SETTINGS, "epochs": epochs})
    rows.append({"epochs": epochs, **{c: f"{result['summary'][c]['final_mean_accuracy']['mean']:.3f}" for c in CONDITIONS}})
print("final mean accuracy (3 seeds)")
print(metrics.format_table(rows))

## Split MNIST (16-D random projection)

In [ ]:
have_mnist = all((MNIST_DIR / name).is_file() for name in FILES.values())
if not have_mnist:
    print(f"MNIST archives not found in {MNIST_DIR.resolve()}; skipping. To download them run from the repository root:\n"
          "  python experiments/class_incremental.py --dataset split_mnist --output results/split_mnist.json")
else:
    mnist = make_feature_data(*load_projected_mnist(MNIST_DIR, projection_seed=0, download=False))
    mnist_result = run_comparison(mnist, SEEDS, **SETTINGS)
    summary_table(mnist_result)
    plot_matrices(mnist_result, "Split MNIST")

In [ ]:
# Budget sensitivity on one seed; the polynomial features computed above are reused.
if have_mnist:
    tasks, memory = make_task_plan(mnist.train_y, 0, MEMORY_PER_CLASS)
    rows = []
    for epochs in (1, 3, 10, 30):
        runs = {c: run_condition(c, mnist, tasks, memory, seed=0, **{**TRAIN, "epochs": epochs})
                for c in CONDITIONS}
        rows.append({"epochs": epochs, **{c: f"{runs[c]['final_mean_accuracy']:.3f}" for c in CONDITIONS}})
    print("final mean accuracy (seed 0)")
    print(metrics.format_table(rows))

### Reading the results

Compare means and standard deviations across paired seeds, not a single run or a single heatmap. Always check the
epoch tables: none of these runs is trained to convergence, so absolute numbers depend strongly on the budget.
`expanding` versus `fixed_linear` measures the value of adding capacity; `expanding` versus `fixed_capacity`
measures the value of adding it gradually. Seeds change only task-internal shuffling and replay selection (the
data, test sets and projection are fixed), so the across-seed spread understates dataset-level variability.

## Update rules on Split MNIST (`expanding` schedule)

The binary notebook compares update rules (`frozen_old`, `small_old_lr`, Strategy B) only on the sinusoid task. Here the
same rules are applied to the softmax head. A coefficient is *old* if its feature row existed before this experience and
its class column was already introduced; new rows and the new classes' columns always train. Strategy B projects every
gradient onto the null space of the design at 3 stored inputs per old class, so logits at those inputs cannot change.

Columns beyond accuracy are the learning indicators: `loss_reduction` (training loss before minus after, averaged over
experiences), `old_logit_change` / `old_flip_fraction` (how much old-class logits and predictions moved on old-class test
inputs). `null_dim` only exists for Strategy B.

In [ ]:
from polynomial_cl.class_incremental import STRATEGIES, run_strategy_comparison

if have_mnist:
    strategy_result = run_strategy_comparison(mnist, SEEDS, **SETTINGS)
    summary_table(strategy_result)
    first = {r["strategy"]: r for r in strategy_result["runs"] if r["seed"] == 0}
    print("\nseed 0, per experience: training loss before -> after | trainable coefficients | null_dim")
    for name, run in first.items():
        cells = [f"{b:.2f}->{a:.2f} | {n} | {nd}" for (b, a), n, nd in
                 zip(run["train_loss_by_task"], run["n_trainable_by_task"], run["null_dim_by_task"])]
        print(f"{name:20s}", "  ".join(cells))

### Strategy B is a stability-plasticity dial

More reference inputs per old class mean less forgetting but less learning; with every stored example used as a
reference the null space is empty and nothing can be learned. The best setting is not known in advance (and picking it
on the test data would be circular), so read this as a trade-off curve, not a tuned result.

In [ ]:
if have_mnist:
    rows = []
    for n_ref in (1, 3, 10, 30, 100, 200):
        res = run_strategy_comparison(mnist, SEEDS[:3], strategies=("strategy_b",), n_ref_per_class=n_ref, **SETTINGS)
        s, runs_b = res["summary"]["strategy_b"], res["runs"]
        rows.append({
            "refs_per_class": n_ref,
            "final_mean_acc": f"{s['final_mean_accuracy']['mean']:.3f}",
            "final_current_acc": f"{s['final_current_accuracy']['mean']:.3f}",
            "forgetting": f"{s['mean_forgetting']['mean']:.3f}",
            "null_dim_last_exp": int(np.mean([r["null_dim_by_task"][-1] for r in runs_b])),
        })
    print(metrics.format_table(rows))